# Training Analysis Notebook

This notebook provides tools for analyzing training metrics, comparing
benchmark results, and visualizing GPU utilization across different
fine-tuning configurations (LoRA, QLoRA, DeepSpeed ZeRO-3).

In [ ]:
import json
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Training Metrics

In [ ]:
def load_jsonl(path: str) -> pd.DataFrame:
    """Load a JSONL file into a DataFrame."""
    records = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return pd.DataFrame(records)


# Load training metrics (update paths as needed)
metrics_files = glob.glob("outputs/*/training_metrics.jsonl")
print(f"Found {len(metrics_files)} training metrics files:")
for f in metrics_files:
    print(f"  {f}")

## 2. Training Loss Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

for path in metrics_files:
    config_name = Path(path).parent.name
    df = load_jsonl(path)
    if "loss" in df.columns and "step" in df.columns:
        ax.plot(df["step"], df["loss"], label=config_name, alpha=0.8)

ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Comparison")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Throughput Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for path in metrics_files:
    config_name = Path(path).parent.name
    df = load_jsonl(path)
    if "throughput_tokens_per_sec" in df.columns:
        ax1.plot(
            df["step"], df["throughput_tokens_per_sec"], label=config_name, alpha=0.8
        )
    if "step_time_sec" in df.columns:
        ax2.plot(df["step"], df["step_time_sec"], label=config_name, alpha=0.8)

ax1.set_xlabel("Step")
ax1.set_ylabel("Tokens/sec")
ax1.set_title("Training Throughput")
ax1.legend()

ax2.set_xlabel("Step")
ax2.set_ylabel("Step Time (sec)")
ax2.set_title("Step Latency")
ax2.legend()

plt.tight_layout()
plt.show()

## 4. GPU Memory Analysis

In [ ]:
gpu_files = glob.glob("outputs/*/gpu_metrics.jsonl")

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

for path in gpu_files:
    config_name = Path(path).parent.name
    df = load_jsonl(path)
    if "gpu_memory_allocated_mb" in df.columns:
        ax.plot(
            df.index,
            df["gpu_memory_allocated_mb"] / 1024,
            label=f"{config_name} (allocated)",
            alpha=0.8,
        )
    if "gpu_max_memory_allocated_mb" in df.columns:
        peak = df["gpu_max_memory_allocated_mb"].max() / 1024
        ax.axhline(
            y=peak,
            linestyle="--",
            alpha=0.5,
            label=f"{config_name} peak: {peak:.1f} GB",
        )

ax.set_xlabel("Measurement Index")
ax.set_ylabel("GPU Memory (GB)")
ax.set_title("GPU Memory Usage During Training")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Benchmark Comparison Table

In [ ]:
benchmark_files = glob.glob("artifacts/benchmarks/benchmark_*.json")

if benchmark_files:
    results = []
    for path in benchmark_files:
        with open(path) as f:
            results.append(json.load(f))

    df = pd.DataFrame(results)
    display_cols = [
        "config_name",
        "method",
        "num_gpus",
        "effective_batch_size",
        "total_training_time_sec",
        "tokens_per_sec",
        "gpu_memory_peak_mb",
        "training_loss_final",
    ]
    available_cols = [c for c in display_cols if c in df.columns]
    print(df[available_cols].to_string(index=False))
else:
    print("No benchmark results found. Run: bash scripts/benchmark_training.sh")

## 6. GPU Utilization Over Time

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

for path in gpu_files:
    config_name = Path(path).parent.name
    df = load_jsonl(path)
    if "gpu_utilization_pct" in df.columns:
        valid = df[df["gpu_utilization_pct"].notna()]
        if not valid.empty:
            ax.plot(
                valid.index, valid["gpu_utilization_pct"], label=config_name, alpha=0.8
            )

ax.set_xlabel("Measurement Index")
ax.set_ylabel("GPU Utilization (%)")
ax.set_title("GPU Utilization During Training")
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()